# Meaningful Variations

In a respondent population of 15, much much does going from a 4.2 to a 4.4 average score matter? 

This notebook examines this question and quantifies the difference. It also looks for dicontinous jumps average scores when there is little change to the distribution of scores. This is a **Discontinuity or Jump** analysis. It also looks the reverse of this: when a small change in distribution of scores can create a large change in the average; for example, one person switching their score from 2 to 3. This is a **Leverage or Sensitivity** analysis. 

In [11]:
from math import factorial
import os

def count_permutations_for_distribution(distribution, n):
    """Calculate number of permutations for a given distribution using multinomial coefficient"""
    numerator = factorial(n)
    denominator = 1
    for count in distribution:
        denominator *= factorial(count)
    return numerator // denominator

def find_distributions_with_average(n_respondents, target_average, scores=[1,2,3,4,5]):
    """Find all score distributions that achieve target average"""
    
    # Check if average is achievable
    target_sum = target_average * n_respondents
    if target_sum != int(target_sum):
        return None, f"Average {target_average} is not possible with {n_respondents} respondents (sum must be integer)"
    
    target_sum = int(target_sum)
    
    # Check if sum is within valid range
    min_possible_sum = n_respondents * min(scores)
    max_possible_sum = n_respondents * max(scores)
    
    if target_sum < min_possible_sum or target_sum > max_possible_sum:
        return None, f"Average {target_average} is not possible (sum {target_sum} out of range [{min_possible_sum}, {max_possible_sum}])"
    
    valid_distributions = []
    
    # This is hard coded for a list of possible scores, in this case
    ## [1, 2, 3, 4, 5]
    for n1 in range(n_respondents + 1):
        for n2 in range(n_respondents - n1 + 1):
            for n3 in range(n_respondents - n1 - n2 + 1):
                for n4 in range(n_respondents - n1 - n2 - n3 + 1):
                    n5 = n_respondents - n1 - n2 - n3 - n4
                    if n1*1 + n2*2 + n3*3 + n4*4 + n5*5 == target_sum:
                        valid_distributions.append((n1, n2, n3, n4, n5))
    
    return valid_distributions, None

def calculate_average_distribution(n_respondents, target_average):
    """Calculate the average counts of 1s, 2s, 3s, 4s, and 5s across all distributions"""
    
    result = find_distributions_with_average(n_respondents, target_average)
    
    if result[1] is not None:  # Error message exists
        return None, result[1]
    
    distributions = result[0]
    
    # Calculate weighted average of each score count
    # Weight by the number of permutations each distribution generates
    total_weight = 0
    weighted_sums = [0, 0, 0, 0, 0]  # for scores 1, 2, 3, 4, 5
    
    for dist in distributions:
        weight = count_permutations_for_distribution(dist, n_respondents)
        total_weight += weight
        
        for i in range(5):
            weighted_sums[i] += dist[i] * weight
    
    # Calculate weighted averages
    avg_counts = [ws / total_weight for ws in weighted_sums]
    
    return avg_counts, None

def compare_two_averages(n_respondents, avg1, avg2):
    """Compare the average distributions for two different average scores"""
    
    def find_closest_valid_averages(n, target_avg):
        """Find the closest valid averages below and above target"""
        min_sum = n  # all 1s
        max_sum = 5 * n  # all 5s
        
        # Generate all possible averages for this number of respondents
        possible_averages = []
        for total_sum in range(min_sum, max_sum + 1):
            avg = total_sum / n
            possible_averages.append(avg)
        
        # Find closest below and above target
        lower = None
        higher = None
        
        for avg in possible_averages:
            if avg < target_avg:
                if lower is None or avg > lower:
                    lower = avg
            elif avg > target_avg:
                if higher is None or avg < higher:
                    higher = avg
        
        return lower, higher
    
    print(f"Comparing average {avg1} vs {avg2} with {n_respondents} respondents")
    print("=" * 70)
    
    # Get average distribution for first average
    result1 = calculate_average_distribution(n_respondents, avg1)
    if result1[1] is not None:
        print(f"\nAverage {avg1} is not possible: {result1[1]}")
        lower, higher = find_closest_valid_averages(n_respondents, avg1)
        if lower is not None:
            print(f"  → Closest lower average: {lower:.6g}")
        if higher is not None:
            print(f"  → Closest higher average: {higher:.6g}")
        avg_dist1 = None
    else:
        avg_dist1 = result1[0]
    
    # Get average distribution for second average
    result2 = calculate_average_distribution(n_respondents, avg2)
    if result2[1] is not None:
        print(f"\nAverage {avg2} is not possible: {result2[1]}")
        lower, higher = find_closest_valid_averages(n_respondents, avg2)
        if lower is not None:
            print(f"  → Closest lower average: {lower:.6g}")
        if higher is not None:
            print(f"  → Closest higher average: {higher:.6g}")
        avg_dist2 = None
    else:
        avg_dist2 = result2[0]
    
    # Only continue with comparison if both averages are valid
    if avg_dist1 is None or avg_dist2 is None:
        print("\nCannot complete comparison - one or both averages are not possible.")
        return
    
    # Display results
    print(f"\nAverage counts across all distributions:\n")
    print(f"{'Score':<8} {'Avg ' + str(avg1):<12} {'%':<8} {'Avg ' + str(avg2):<12} {'%':<8} {'Diff':<12} {'Diff %':<10}")
    print("-" * 80)
    
    for score in range(1, 6):
        count1 = avg_dist1[score-1]
        count2 = avg_dist2[score-1]
        pct1 = (count1 / n_respondents) * 100
        pct2 = (count2 / n_respondents) * 100
        diff = count2 - count1
        diff_pct = (diff / n_respondents) * 100
        print(f"{score:<8} {count1:<12.4f} {pct1:<8.2f} {count2:<12.4f} {pct2:<8.2f} {diff:+12.4f} {diff_pct:+10.2f}")
    
    print(f"\nTotal:   {sum(avg_dist1):<12.4f} {100.0:<8.2f} {sum(avg_dist2):<12.4f} {100.0:<8.2f}")
    
    # Summary
    print("\n" + "=" * 80)
    print("Interpretation:")
    print(f"Moving from average {avg1} to {avg2}:")
    for score in range(1, 6):
        diff = avg_dist2[score-1] - avg_dist1[score-1]
        if abs(diff) > 0.01:  # Only show meaningful differences
            direction = "more" if diff > 0 else "fewer"
            diff_pct = (diff / n_respondents) * 100
            print(f"  • {direction} {score}s (difference: {diff:+.4f}, {diff_pct:+.2f}%)")

In [15]:
compare_two_averages(5, 3.2, 5)

Comparing average 3.2 vs 5 with 5 respondents

Average counts across all distributions:

Score    Avg 3.2      %        Avg 5        %        Diff         Diff %    
--------------------------------------------------------------------------------
1        0.7123       14.25    0.0000       0.00          -0.7123     -14.25
2        0.9315       18.63    0.0000       0.00          -0.9315     -18.63
3        1.0959       21.92    0.0000       0.00          -1.0959     -21.92
4        1.1644       23.29    0.0000       0.00          -1.1644     -23.29
5        1.0959       21.92    5.0000       100.00        +3.9041     +78.08

Total:   5.0000       100.00   5.0000       100.00  

Interpretation:
Moving from average 3.2 to 5:
  • fewer 1s (difference: -0.7123, -14.25%)
  • fewer 2s (difference: -0.9315, -18.63%)
  • fewer 3s (difference: -1.0959, -21.92%)
  • fewer 4s (difference: -1.1644, -23.29%)
  • more 5s (difference: +3.9041, +78.08%)


In [17]:
def analyze_average_range(n_respondents, start_avg=3.0, end_avg=5.0, step_size=0.1):
    """
    Analyze differences in distribution counts across a range of average scores.
    
    Parameters:
    -----------
    n_respondents : int
        Number of respondents in the population
    start_avg : float
        Starting average score (default: 3.0)
    end_avg : float
        Ending average score (default: 5.0)
    step_size : float
        Step size for the range (default: 0.1)
    
    Returns:
    --------
    dict : Dictionary containing population size, step size, and pairwise comparisons
           Each comparison contains list of tuples (delta, percent) for each score
    """
    
    # Generate the range of averages
    # Use round to avoid floating point precision issues
    num_steps = int(round((end_avg - start_avg) / step_size)) + 1
    averages = [round(start_avg + i * step_size, 10) for i in range(num_steps)]
    
    # Initialize result dictionary
    result = {
        "population_size": n_respondents,
        "step_size": step_size,
        "start_average": start_avg,
        "end_average": end_avg,
        "comparisons": {}
    }
    
    # Calculate differences for each consecutive pair
    for i in range(len(averages) - 1):
        avg1 = averages[i]
        avg2 = averages[i + 1]
        
        # Create key for this pair
        pair_key = f"({avg1:.1f}, {avg2:.1f})"
        
        # Get distribution for first average
        dist1_result = calculate_average_distribution(n_respondents, avg1)
        
        # Get distribution for second average
        dist2_result = calculate_average_distribution(n_respondents, avg2)
        
        # Check if either average is not possible
        if dist1_result[1] is not None or dist2_result[1] is not None:
            result["comparisons"][pair_key] = "N/A"
        else:
            # Calculate differences: avg2 - avg1
            avg_dist1 = dist1_result[0]
            avg_dist2 = dist2_result[0]
            
            # Store both delta and percentage for each score
            differences_with_percent = []
            for j in range(5):
                delta = avg_dist2[j] - avg_dist1[j]
                percent = (delta / n_respondents) * 100
                differences_with_percent.append((delta, percent))
            
            result["comparisons"][pair_key] = differences_with_percent
    
    return result


def print_range_analysis(result):
    """
    Pretty print the results from analyze_average_range
    
    Parameters:
    -----------
    result : dict
        Output from analyze_average_range function
    """
    print(f"Population Size: {result['population_size']}")
    print(f"Average Range: {result['start_average']} to {result['end_average']}")
    print(f"Step Size: {result['step_size']}")
    print("=" * 140)
    print(f"\n{'Pair':<20} {'Δ Score 1 (% pop)':<25} {'Δ Score 2 (% pop)':<25} {'Δ Score 3 (% pop)':<25} {'Δ Score 4 (% pop)':<25} {'Δ Score 5 (% pop)':<25}")
    print("-" * 140)
    
    for pair, differences in result["comparisons"].items():
        if differences == "N/A":
            print(f"{pair:<20} {'N/A':<25}")
        else:
            # Format each delta with its percentage
            formatted_values = []
            for delta, percent in differences:
                formatted_values.append(f"{delta:+7.4f} ({percent:+6.2f}%)")
            
            diff_str = "".join([f"{val:<25}" for val in formatted_values])
            print(f"{pair:<20}{diff_str}")
    
    print("=" * 140)

In [38]:
# Example usage
print("Example 1: Population size 20, default range (3.0 to 5.0), step 0.1")
result1 = analyze_average_range(20, step_size=0.1)
print_range_analysis(result1)

print("\n\n")

print("Example 2: Population size 5, range 3.0 to 4.0, step 0.2")
result2 = analyze_average_range(5, start_avg=3.0, end_avg=4.0, step_size=0.2)
print_range_analysis(result2)

print("\n\n")

print("Example 3: Population size 10, range 3.5 to 4.5, step 0.25")
result3 = analyze_average_range(10, start_avg=3.5, end_avg=4.5, step_size=0.25)
print_range_analysis(result3)

Example 1: Population size 20, default range (3.0 to 5.0), step 0.1
Population Size: 20
Average Range: 3.0 to 5.0
Step Size: 0.1

Pair                 Δ Score 1 (% pop)         Δ Score 2 (% pop)         Δ Score 3 (% pop)         Δ Score 4 (% pop)         Δ Score 5 (% pop)        
--------------------------------------------------------------------------------------------------------------------------------------------
(3.0, 3.1)          -0.3871 ( -1.94%)        -0.2107 ( -1.05%)        -0.0101 ( -0.05%)        +0.2009 ( +1.00%)        +0.4070 ( +2.04%)        
(3.1, 3.2)          -0.3677 ( -1.84%)        -0.2194 ( -1.10%)        -0.0303 ( -0.15%)        +0.1899 ( +0.95%)        +0.4276 ( +2.14%)        
(3.2, 3.3)          -0.3487 ( -1.74%)        -0.2271 ( -1.14%)        -0.0507 ( -0.25%)        +0.1776 ( +0.89%)        +0.4490 ( +2.24%)        
(3.3, 3.4)          -0.3299 ( -1.65%)        -0.2338 ( -1.17%)        -0.0714 ( -0.36%)        +0.1637 ( +0.82%)        +0.4713 ( +2.36%)   

## Create a comprehensive test suite that checks the following: 

 - ✅ Multinomial coefficient calculation - verifies the permutation counting is correct
 - ✅ Valid average scores - ensures distributions sum correctly and achieve target
 - ✅ Impossible averages (non-integer sums) - verifies error handling
 - ✅ Out of range averages - verifies bounds checking
 - ✅ Weighted average calculation - ensures distribution counts sum to n
 - ✅ Edge cases - minimum (1.0) and maximum (5.0) averages
 - ✅ Range analysis structure - verifies correct number of comparisons
 - ✅ Delta sum verification - ensures deltas sum to zero (conservation)
 - ✅ N/A handling - tests with population size where many averages are impossible
 - ✅ Percentage calculations - verifies percentages match expected values

In [40]:
# ============================================================================
# COMPREHENSIVE TEST SUITE
# ============================================================================

def run_tests():
    """Run comprehensive tests on all functions"""
    
    print("=" * 80)
    print("RUNNING COMPREHENSIVE TEST SUITE")
    print("=" * 80)
    
    # Test 1: count_permutations_for_distribution
    print("\n--- TEST 1: count_permutations_for_distribution ---")
    test_dist = (2, 1, 0, 1, 1)  # 5 respondents
    n = 5
    result = count_permutations_for_distribution(test_dist, n)
    expected = factorial(5) // (factorial(2) * factorial(1) * factorial(0) * factorial(1) * factorial(1))
    print(f"Distribution: {test_dist}, n={n}")
    print(f"Permutations: {result}")
    print(f"Expected: {expected}")
    print(f"✓ PASS" if result == expected else f"✗ FAIL")
    
    # Test 2: find_distributions_with_average - valid average
    print("\n--- TEST 2: find_distributions_with_average (valid) ---")
    n = 5
    avg = 3.0
    distributions, error = find_distributions_with_average(n, avg)
    print(f"n={n}, average={avg}")
    print(f"Number of distributions found: {len(distributions) if distributions else 0}")
    print(f"Error: {error}")
    # Verify all distributions sum to n and achieve target
    if distributions:
        all_valid = True
        target_sum = int(avg * n)
        for dist in distributions[:3]:  # Check first 3
            dist_sum = sum(dist)
            score_sum = sum((i+1) * count for i, count in enumerate(dist))
            if dist_sum != n or score_sum != target_sum:
                all_valid = False
            print(f"  {dist} -> sum={dist_sum}, score_sum={score_sum}")
        print(f"✓ PASS" if all_valid and error is None else f"✗ FAIL")
    
    # Test 3: find_distributions_with_average - impossible average
    print("\n--- TEST 3: find_distributions_with_average (impossible) ---")
    n = 5
    avg = 3.15  # Not possible with 5 respondents (sum would be 15.75)
    distributions, error = find_distributions_with_average(n, avg)
    print(f"n={n}, average={avg}")
    print(f"Error message: {error}")
    print(f"✓ PASS" if error is not None and distributions is None else f"✗ FAIL")
    
    # Test 4: find_distributions_with_average - out of range
    print("\n--- TEST 4: find_distributions_with_average (out of range) ---")
    n = 5
    avg = 6.0  # Impossible (max is 5.0)
    distributions, error = find_distributions_with_average(n, avg)
    print(f"n={n}, average={avg}")
    print(f"Error message: {error}")
    print(f"✓ PASS" if error is not None and distributions is None else f"✗ FAIL")
    
    # Test 5: calculate_average_distribution
    print("\n--- TEST 5: calculate_average_distribution ---")
    n = 10
    avg = 3.0
    avg_dist, error = calculate_average_distribution(n, avg)
    print(f"n={n}, average={avg}")
    if avg_dist:
        print(f"Average distribution: {[f'{x:.4f}' for x in avg_dist]}")
        total = sum(avg_dist)
        print(f"Sum of average counts: {total:.4f} (should equal {n})")
        print(f"✓ PASS" if abs(total - n) < 0.0001 else f"✗ FAIL")
    else:
        print(f"Error: {error}")
        print(f"✗ FAIL")
    
    # Test 6: Edge case - all respondents give same score
    print("\n--- TEST 6: Edge case - all same score (avg=1.0) ---")
    n = 5
    avg = 1.0
    distributions, error = find_distributions_with_average(n, avg)
    print(f"n={n}, average={avg}")
    print(f"Distributions found: {distributions}")
    print(f"Expected: [(5, 0, 0, 0, 0)]")
    print(f"✓ PASS" if distributions == [(5, 0, 0, 0, 0)] else f"✗ FAIL")
    
    # Test 7: Edge case - all respondents give max score
    print("\n--- TEST 7: Edge case - all same score (avg=5.0) ---")
    n = 5
    avg = 5.0
    distributions, error = find_distributions_with_average(n, avg)
    print(f"n={n}, average={avg}")
    print(f"Distributions found: {distributions}")
    print(f"Expected: [(0, 0, 0, 0, 5)]")
    print(f"✓ PASS" if distributions == [(0, 0, 0, 0, 5)] else f"✗ FAIL")
    
    # Test 8: analyze_average_range - basic functionality
    print("\n--- TEST 8: analyze_average_range - basic functionality ---")
    n = 10
    result = analyze_average_range(n, start_avg=3.0, end_avg=3.5, step_size=0.1)
    print(f"n={n}, range=[3.0, 3.5], step=0.1")
    print(f"Number of comparisons: {len(result['comparisons'])}")
    print(f"Expected: 5 comparisons")
    
    # Check structure
    first_key = list(result['comparisons'].keys())[0]
    first_value = result['comparisons'][first_key]
    if first_value != "N/A":
        print(f"First comparison: {first_key}")
        print(f"  Data structure: list of {len(first_value)} tuples")
        print(f"  Sample: {first_value[0]} (delta, percent)")
        
        # Verify percentages add up correctly (should sum to 0)
        delta_sum = sum(delta for delta, _ in first_value)
        percent_sum = sum(percent for _, percent in first_value)
        print(f"  Sum of deltas: {delta_sum:.6f} (should be ~0)")
        print(f"  Sum of percents: {percent_sum:.6f} (should be ~0)")
        print(f"✓ PASS" if abs(delta_sum) < 0.0001 and abs(percent_sum) < 0.0001 else f"✗ FAIL")
    
    # Test 9: analyze_average_range - with impossible averages
    print("\n--- TEST 9: analyze_average_range - with impossible averages ---")
    n = 3
    result = analyze_average_range(n, start_avg=3.0, end_avg=3.5, step_size=0.1)
    print(f"n={n}, range=[3.0, 3.5], step=0.1")
    na_count = sum(1 for v in result['comparisons'].values() if v == "N/A")
    print(f"Number of N/A results: {na_count}")
    print(f"Sample comparisons:")
    for i, (key, value) in enumerate(result['comparisons'].items()):
        if i < 3:
            print(f"  {key}: {value if value == 'N/A' else 'Valid'}")
    print(f"✓ PASS (found some N/A as expected for n=3)")
    
    # Test 10: Verify percentage calculations
    print("\n--- TEST 10: Verify percentage calculations ---")
    n = 20
    result = analyze_average_range(n, start_avg=4.0, end_avg=4.2, step_size=0.1)
    first_valid_pair = None
    for key, value in result['comparisons'].items():
        if value != "N/A":
            first_valid_pair = (key, value)
            break
    
    if first_valid_pair:
        key, values = first_valid_pair
        print(f"Pair: {key}, n={n}")
        for i, (delta, percent) in enumerate(values):
            expected_percent = (delta / n) * 100
            print(f"  Score {i+1}: delta={delta:.4f}, percent={percent:.4f}%, expected={expected_percent:.4f}%")
            if abs(percent - expected_percent) > 0.001:
                print(f"  ✗ FAIL - percentage mismatch!")
                break
        else:
            print(f"✓ PASS - all percentages correct")
    
    print("\n" + "=" * 80)
    print("TEST SUITE COMPLETE")
    print("=" * 80)

In [41]:
# ============================================================================
# RUN TESTS
# ============================================================================

if __name__ == "__main__":
    run_tests()
    
    # Run some example analyses
    print("\n\n")
    print("=" * 80)
    print("EXAMPLE ANALYSES")
    print("=" * 80)
    
    print("\n\nExample 1: Population size 20, range 3.0-4.0, step 0.2")
    result1 = analyze_average_range(20, start_avg=3.0, end_avg=4.0, step_size=0.2)
    print_range_analysis(result1)
    
    print("\n\nExample 2: Population size 10, range 4.0-5.0, step 0.25")
    result2 = analyze_average_range(10, start_avg=4.0, end_avg=5.0, step_size=0.25)
    print_range_analysis(result2)

RUNNING COMPREHENSIVE TEST SUITE

--- TEST 1: count_permutations_for_distribution ---
Distribution: (2, 1, 0, 1, 1), n=5
Permutations: 60
Expected: 60
✓ PASS

--- TEST 2: find_distributions_with_average (valid) ---
n=5, average=3.0
Number of distributions found: 12
Error: None
  (0, 0, 5, 0, 0) -> sum=5, score_sum=15
  (0, 1, 3, 1, 0) -> sum=5, score_sum=15
  (0, 2, 1, 2, 0) -> sum=5, score_sum=15
✓ PASS

--- TEST 3: find_distributions_with_average (impossible) ---
n=5, average=3.15
Error message: Average 3.15 is not possible with 5 respondents (sum must be integer)
✓ PASS

--- TEST 4: find_distributions_with_average (out of range) ---
n=5, average=6.0
Error message: Average 6.0 is not possible (sum 30 out of range [5, 25])
✓ PASS

--- TEST 5: calculate_average_distribution ---
n=10, average=3.0
Average distribution: ['1.8973', '2.0506', '2.1043', '2.0506', '1.8973']
Sum of average counts: 10.0000 (should equal 10)
✓ PASS

--- TEST 6: Edge case - all same score (avg=1.0) ---
n=5, avera

## Create a table to display the results

In [19]:
def build_full_comparison_matrix(n_respondents, start_avg=3.0, end_avg=5.0, step_size=0.1):
    """
    Build a full comparison matrix for all pairs of averages in the range.
    
    Parameters:
    -----------
    n_respondents : int
        Number of respondents in the population
    start_avg : float
        Starting average score (default: 3.0)
    end_avg : float
        Ending average score (default: 5.0)
    step_size : float
        Step size for the range (default: 0.1)
    
    Returns:
    --------
    dict : Dictionary containing:
        - population_size
        - averages: list of all average values
        - matrix: 2D dictionary where matrix[avg1][avg2] = comparison data
    """
    
    # Generate the range of averages
    num_steps = int(round((end_avg - start_avg) / step_size)) + 1
    averages = [round(start_avg + i * step_size, 10) for i in range(num_steps)]
    
    # Initialize result dictionary
    result = {
        "population_size": n_respondents,
        "step_size": step_size,
        "start_average": start_avg,
        "end_average": end_avg,
        "averages": averages,
        "matrix": {}
    }
    
    # Build comparison matrix for all pairs
    for avg1 in averages:
        result["matrix"][avg1] = {}
        for avg2 in averages:
            if avg1 == avg2:
                # Diagonal - no difference
                result["matrix"][avg1][avg2] = {
                    "differences": [(0.0, 0.0)] * 5,
                    "type": "diagonal"
                }
            else:
                # Get distributions
                dist1_result = calculate_average_distribution(n_respondents, avg1)
                dist2_result = calculate_average_distribution(n_respondents, avg2)
                
                # Check if either average is not possible
                if dist1_result[1] is not None or dist2_result[1] is not None:
                    result["matrix"][avg1][avg2] = {
                        "differences": "N/A",
                        "type": "impossible"
                    }
                else:
                    # Calculate differences: avg2 - avg1
                    avg_dist1 = dist1_result[0]
                    avg_dist2 = dist2_result[0]
                    
                    differences_with_percent = []
                    for j in range(5):
                        delta = avg_dist2[j] - avg_dist1[j]
                        percent = (delta / n_respondents) * 100
                        differences_with_percent.append((delta, percent))
                    
                    result["matrix"][avg1][avg2] = {
                        "differences": differences_with_percent,
                        "type": "valid"
                    }
    
    return result

## Visualization

In [21]:
# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

from prettytable import PrettyTable, HRuleStyle, VRuleStyle
import numpy as np

# For image generation
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Optional: for colored terminal output
try:
    from colorama import Fore, Back, Style, init
    init(autoreset=True)
    COLORAMA_AVAILABLE = True
except ImportError:
    COLORAMA_AVAILABLE = False

def calculate_color_score(differences_data):
    """
    Calculate a color score based on changes in scores 4 and 5.
    
    Parameters:
    -----------
    differences_data : list of tuples or "N/A"
        The differences data for a cell
    
    Returns:
    --------
    float or None : Score value (positive = improvement, negative = decline)
    """
    if differences_data == "N/A":
        return None
    
    # Weight score 4 and 5 changes
    delta_4 = differences_data[3][0]  # Score 4 delta
    delta_5 = differences_data[4][0]  # Score 5 delta
    
    # Give more weight to 5s than 4s
    score = delta_4 * 1.0 + delta_5 * 1.5
    
    return score


def get_cell_color_hex(score, max_score):
    """
    Get HTML hex color based on score.
    
    Parameters:
    -----------
    score : float or None
        Color score (None for N/A cells)
    max_score : float
        Maximum absolute score for normalization
    
    Returns:
    --------
    str : HTML color code
    """
    if score is None:
        return "#CCCCCC"  # Gray for N/A
    
    if max_score == 0:
        return "#F0FFF0"  # Very light green for no change
    
    # Normalize score to [-1, 1]
    normalized = score / max_score if max_score > 0 else 0
    
    if abs(normalized) < 0.05:  # Negligible difference
        return "#F0FFF0"  # Very light green
    elif normalized > 0:  # Positive (improvement - more 4s and 5s)
        if normalized < 0.3:
            return "#D4EDDA"  # Very light green
        elif normalized < 0.5:
            return "#B2DFBB"  # Light green
        elif normalized < 0.7:
            return "#70C876"  # Medium green
        elif normalized < 0.9:
            return "#28A745"  # Green
        else:
            return "#1E7E34"  # Dark green
    else:  # Negative (decline - fewer 4s and 5s)
        abs_norm = abs(normalized)
        if abs_norm < 0.3:
            return "#F8D7DA"  # Very light red
        elif abs_norm < 0.5:
            return "#F1B0B7"  # Light red
        elif abs_norm < 0.7:
            return "#E4717A"  # Medium red
        elif abs_norm < 0.9:
            return "#DC3545"  # Red
        else:
            return "#BD2130"  # Dark red


def create_comparison_table_html(matrix_data, 
                                 output_file=os.path.join("..", "html", "comparison_table.html")
                                ):
    """
    Create an HTML table visualization of the comparison matrix.
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    output_file : str
        Output filename for HTML file
    
    Returns:
    --------
    str : Path to created file
    """
    averages = matrix_data["averages"]
    matrix = matrix_data["matrix"]
    n = matrix_data["population_size"]
    
    # Calculate max score for normalization
    max_score = 0
    for avg1 in averages:
        for avg2 in averages:
            cell_data = matrix[avg1][avg2]
            if cell_data["type"] == "valid":
                score = calculate_color_score(cell_data["differences"])
                if score is not None:
                    max_score = max(max_score, abs(score))
    
    # Build HTML
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Score Distribution Comparison Matrix</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 20px;
                background-color: #f5f5f5;
            }}
            h1 {{
                color: #333;
            }}
            .info {{
                margin-bottom: 20px;
                padding: 10px;
                background-color: white;
                border-radius: 5px;
            }}
            table {{
                border-collapse: collapse;
                background-color: white;
                box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }}
            th, td {{
                border: 1px solid #ddd;
                padding: 8px;
                text-align: center;
                min-width: 80px;
            }}
            th {{
                background-color: #4CAF50;
                color: white;
                font-weight: bold;
            }}
            .row-header {{
                background-color: #4CAF50;
                color: white;
                font-weight: bold;
            }}
            .diagonal {{
                background-color: #E8E8E8;
                color: #999;
            }}
            .legend {{
                margin-top: 20px;
                padding: 15px;
                background-color: white;
                border-radius: 5px;
            }}
            .legend-item {{
                display: inline-block;
                margin-right: 20px;
                margin-bottom: 10px;
            }}
            .legend-color {{
                display: inline-block;
                width: 30px;
                height: 20px;
                margin-right: 5px;
                vertical-align: middle;
                border: 1px solid #ccc;
            }}
            .cell-content {{
                font-size: 0.85em;
            }}
        </style>
    </head>
    <body>
        <h1>Score Distribution Comparison Matrix</h1>
        <div class="info">
            <strong>Population Size:</strong> {n}<br>
            <strong>Average Range:</strong> {matrix_data['start_average']} to {matrix_data['end_average']}<br>
            <strong>Step Size:</strong> {matrix_data['step_size']}<br>
            <strong>Color Coding:</strong> Based on changes in counts of 4s and 5s<br>
            <strong>How to Read:</strong> Each cell shows the change from row average (left) to column average (top)
        </div>
        
        <table>
            <tr>
                <th>From ↓ / To →</th>
    """
    
    # Column headers
    for avg in averages:
        html += f"<th>{avg:.1f}</th>"
    html += "</tr>\n"
    
    # Data rows
    for avg1 in averages:
        html += f"<tr><td class='row-header'>{avg1:.1f}</td>"
        
        for avg2 in averages:
            cell_data = matrix[avg1][avg2]
            
            if cell_data["type"] == "diagonal":
                html += "<td class='diagonal'>—</td>"
            elif cell_data["type"] == "impossible":
                html += "<td style='background-color: #CCCCCC;'>N/A</td>"
            else:
                differences = cell_data["differences"]
                score = calculate_color_score(differences)
                color = get_cell_color_hex(score, max_score)
                
                # Format cell content - show Δ for scores 4 and 5 with percentages
                delta_4, pct_4 = differences[3]
                delta_5, pct_5 = differences[4]
                
                cell_html = f"""<td style='background-color: {color};' title='Score 4: {delta_4:+.2f} ({pct_4:+.1f}%), Score 5: {delta_5:+.2f} ({pct_5:+.1f}%)'>
                    <div class='cell-content'>
                        4: {delta_4:+.2f} ({pct_4:+.1f}%)<br>
                        5: {delta_5:+.2f} ({pct_5:+.1f}%)
                    </div>
                </td>"""
                html += cell_html
        
        html += "</tr>\n"
    
    html += """
        </table>
        
        <div class="legend">
            <h3>Legend</h3>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #CCCCCC;"></span>
                <span>N/A (impossible average)</span>
            </div>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #E8E8E8;"></span>
                <span>No change (diagonal)</span>
            </div>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #F0FFF0;"></span>
                <span>Negligible change</span>
            </div>
            <br>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #D4EDDA;"></span>
                <span>Small improvement (more 4s/5s)</span>
            </div>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #70C876;"></span>
                <span>Medium improvement</span>
            </div>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #1E7E34;"></span>
                <span>Large improvement</span>
            </div>
            <br>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #F8D7DA;"></span>
                <span>Small decline (fewer 4s/5s)</span>
            </div>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #E4717A;"></span>
                <span>Medium decline</span>
            </div>
            <div class="legend-item">
                <span class="legend-color" style="background-color: #BD2130;"></span>
                <span>Large decline</span>
            </div>
            <p style="margin-top: 15px; font-size: 0.9em; color: #666;">
                <strong>Note:</strong> Each cell shows the change in average counts of scores 4 and 5 when moving from the row average to the column average.
                Hover over cells to see detailed percentages.
            </p>
        </div>
    </body>
    </html>
    """
    
    # Write to file
    with open(output_file, 'w') as f:
        f.write(html)
    
    print(f"✓ HTML table saved to {output_file}")
    return output_file


def create_comparison_table_image(matrix_data, 
                                  output_file=os.path.join("..", "images", "comparison_table.png"), 
                                  dpi=150, 
                                  fig_width=None, 
                                  fig_height=None, 
                                  cell_size=None):
    """
    Create an image visualization of the comparison matrix using matplotlib.
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    output_file : str
        Output filename for image file (PNG or JPG)
    dpi : int
        Resolution of output image (default: 150)
    fig_width : float or None
        Figure width in inches. If None, automatically calculated (default: None)
    fig_height : float or None
        Figure height in inches. If None, automatically calculated (default: None)
    cell_size : float or None
        Size per cell in inches. Overrides fig_width/height if provided (default: None)
        Recommended values: 0.5-0.8 for many cells, 0.8-1.2 for fewer cells
    
    Returns:
    --------
    str : Path to created file
    """
    averages = matrix_data["averages"]
    matrix = matrix_data["matrix"]
    n = matrix_data["population_size"]
    
    num_avgs = len(averages)
    
    # Calculate max score for normalization
    max_score = 0
    for avg1 in averages:
        for avg2 in averages:
            cell_data = matrix[avg1][avg2]
            if cell_data["type"] == "valid":
                score = calculate_color_score(cell_data["differences"])
                if score is not None:
                    max_score = max(max_score, abs(score))
    
    # Calculate figure size
    if cell_size is not None:
        # Use cell_size to determine dimensions
        fig_width = max(8, num_avgs * cell_size + 2)  # +2 for margins and legend
        fig_height = max(6, num_avgs * cell_size + 2)
    elif fig_width is None or fig_height is None:
        # Automatic scaling based on number of cells
        # Larger tables get proportionally bigger cells for readability
        if num_avgs <= 10:
            base_cell_size = 0.9
        elif num_avgs <= 15:
            base_cell_size = 0.8
        elif num_avgs <= 20:
            base_cell_size = 0.7
        else:
            base_cell_size = 0.6
        
        if fig_width is None:
            fig_width = max(12, num_avgs * base_cell_size + 3)
        if fig_height is None:
            fig_height = max(8, num_avgs * base_cell_size + 2)
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    # Create data matrix for visualization
    color_matrix = np.zeros((num_avgs, num_avgs, 3))
    text_matrix = [['' for _ in range(num_avgs)] for _ in range(num_avgs)]
    
    for i, avg1 in enumerate(averages):
        for j, avg2 in enumerate(averages):
            cell_data = matrix[avg1][avg2]
            
            if cell_data["type"] == "diagonal":
                color_matrix[i, j] = [0.91, 0.91, 0.91]  # Light gray
                text_matrix[i][j] = "—"
            elif cell_data["type"] == "impossible":
                color_matrix[i, j] = [0.8, 0.8, 0.8]  # Gray
                text_matrix[i][j] = "N/A"
            else:
                differences = cell_data["differences"]
                score = calculate_color_score(differences)
                
                # Get color
                hex_color = get_cell_color_hex(score, max_score)
                # Convert hex to RGB
                rgb = tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (1, 3, 5))
                color_matrix[i, j] = rgb
                
                # Format text with percentages
                delta_4, pct_4 = differences[3]
                delta_5, pct_5 = differences[4]
                text_matrix[i][j] = f"4: {delta_4:+.2f} ({pct_4:+.1f}%)\n5: {delta_5:+.2f} ({pct_5:+.1f}%)"
    
    # Display the matrix
    ax.imshow(color_matrix, aspect='auto')
    
    # Add text to cells
    for i in range(num_avgs):
        for j in range(num_avgs):
            text = text_matrix[i][j]
            # Determine text color based on background
            bg_color = color_matrix[i, j]
            text_color = 'white' if sum(bg_color) < 1.5 else 'black'
            # Adjust font size based on number of averages and text length
            font_size = max(5, 9 - num_avgs * 0.15)
            ax.text(j, i, text, ha='center', va='center', 
                   fontsize=font_size, color=text_color)
    
    # Set ticks and labels
    ax.set_xticks(range(num_avgs))
    ax.set_yticks(range(num_avgs))
    ax.set_xticklabels([f"{avg:.1f}" for avg in averages], fontsize=9)
    ax.set_yticklabels([f"{avg:.1f}" for avg in averages], fontsize=9)
    
    # Labels
    ax.set_xlabel('To Average →', fontsize=11, fontweight='bold')
    ax.set_ylabel('From Average ↓', fontsize=11, fontweight='bold')
    
    # Title
    title_text = (f"Score Distribution Comparison Matrix\n"
                 f"Population: {n} | Range: {matrix_data['start_average']:.1f} to "
                 f"{matrix_data['end_average']:.1f} | Step: {matrix_data['step_size']}")
    ax.set_title(title_text, fontsize=12, fontweight='bold', pad=20)
    
    # Add legend
    legend_elements = [
        mpatches.Patch(facecolor='#CCCCCC', label='N/A (impossible)'),
        mpatches.Patch(facecolor='#F0FFF0', label='Negligible'),
        mpatches.Patch(facecolor='#D4EDDA', label='Small improvement'),
        mpatches.Patch(facecolor='#28A745', label='Large improvement'),
        mpatches.Patch(facecolor='#F8D7DA', label='Small decline'),
        mpatches.Patch(facecolor='#DC3545', label='Large decline')
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), 
             fontsize=9, frameon=True)
    
    plt.tight_layout()
    
    # Save figure
    plt.savefig(output_file, dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Image saved to {output_file}")
    return output_file


def create_comparison_table_prettytable(matrix_data, output_file=None, use_colors=False):
    """
    Create a text-based comparison table using PrettyTable.
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    output_file : str or None
        Output filename for text file (default: None = print to console)
    use_colors : bool
        Use ANSI color codes for terminal output (default: False)
        Only works when output_file is None (console output)
    
    Returns:
    --------
    str : Path to created file or table string
    """
    averages = matrix_data["averages"]
    matrix = matrix_data["matrix"]
    n = matrix_data["population_size"]
    
    # Calculate max score for normalization
    max_score = 0
    for avg1 in averages:
        for avg2 in averages:
            cell_data = matrix[avg1][avg2]
            if cell_data["type"] == "valid":
                score = calculate_color_score(cell_data["differences"])
                if score is not None:
                    max_score = max(max_score, abs(score))
    
    # Create PrettyTable
    table = PrettyTable()
    
    # Set column names (header row)
    column_names = ["From↓/To→"] + [f"{avg:.1f}" for avg in averages]
    table.field_names = column_names
    
    # Set alignment
    table.align = "c"  # Center align all columns
    
    # Enable all borders (horizontal and vertical lines between every cell)
    table.hrules = HRuleStyle.ALL  # Horizontal rules between all rows
    table.vrules = VRuleStyle.ALL  # Vertical rules between all columns
    
    # Add rows
    for avg1 in averages:
        row = [f"{avg1:.1f}"]  # Row header
        
        for avg2 in averages:
            cell_data = matrix[avg1][avg2]
            
            if cell_data["type"] == "diagonal":
                cell_text = "—"
            elif cell_data["type"] == "impossible":
                cell_text = "N/A"
            else:
                differences = cell_data["differences"]
                delta_4, pct_4 = differences[3]
                delta_5, pct_5 = differences[4]
                
                # Format cell content
                cell_text = f"4:{delta_4:+.2f}({pct_4:+.1f}%)\n5:{delta_5:+.2f}({pct_5:+.1f}%)"
                
                # Add color if requested and outputting to console
                if use_colors and output_file is None and COLORAMA_AVAILABLE:
                    score = calculate_color_score(differences)
                    if score is not None and max_score > 0:
                        normalized = score / max_score
                        
                        if abs(normalized) < 0.05:
                            # Negligible - no color
                            pass
                        elif normalized > 0:
                            # Positive - green
                            if normalized >= 0.7:
                                cell_text = Fore.GREEN + Style.BRIGHT + cell_text + Style.RESET_ALL
                            else:
                                cell_text = Fore.GREEN + cell_text + Style.RESET_ALL
                        else:
                            # Negative - red
                            if abs(normalized) >= 0.7:
                                cell_text = Fore.RED + Style.BRIGHT + cell_text + Style.RESET_ALL
                            else:
                                cell_text = Fore.RED + cell_text + Style.RESET_ALL
            
            row.append(cell_text)
        
        table.add_row(row)
    
    # Add title and info
    title = f"\nScore Distribution Comparison Matrix\n"
    info = f"Population: {n} | Range: {matrix_data['start_average']:.1f} to {matrix_data['end_average']:.1f} | Step: {matrix_data['step_size']}\n"
    legend = "\nLegend: N/A = impossible average | — = no change (diagonal)\n"
    legend += "Green = improvement (more 4s/5s) | Red = decline (fewer 4s/5s)\n"
    legend += "Each cell shows: Score 4 change (% of pop) and Score 5 change (% of pop)\n"
    
    # Build output
    output = title + info + "\n" + str(table) + legend
    
    if output_file:
        # Save to file
        with open(output_file, 'w', encoding='utf-8') as f:
            # Don't include color codes in file
            if use_colors and COLORAMA_AVAILABLE:
                # Strip color codes for file output
                import re
                output_clean = re.sub(r'\x1b\[[0-9;]*m', '', output)
                f.write(output_clean)
            else:
                f.write(output)
        print(f"✓ Text table saved to {output_file}")
        return output_file
    else:
        # Print to console
        print(output)
        return output


def create_comparison_table(matrix_data, output_file, format='prettytable', 
                           dpi=150, fig_width=None, fig_height=None, cell_size=None,
                           use_colors=False):
    """
    Create a comparison table visualization in the specified format.
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    output_file : str or None
        Output filename (None for console output with prettytable)
    format : str
        Output format: 'prettytable', 'html', or 'image' (default: 'prettytable')
    dpi : int
        Resolution for image output (default: 150)
    fig_width : float or None
        Figure width in inches for image output (default: None = automatic)
    fig_height : float or None
        Figure height in inches for image output (default: None = automatic)
    cell_size : float or None
        Size per cell in inches for image output (default: None = automatic)
        Overrides fig_width/height if provided
    use_colors : bool
        Use ANSI colors for prettytable console output (default: False)
    
    Returns:
    --------
    str : Path to created file or table string
    """
    if format.lower() == 'prettytable':
        return create_comparison_table_prettytable(matrix_data, output_file, use_colors)
    elif format.lower() == 'html':
        return create_comparison_table_html(matrix_data, output_file)
    elif format.lower() in ['image', 'png', 'jpg', 'jpeg']:
        return create_comparison_table_image(matrix_data, output_file, dpi, 
                                             fig_width, fig_height, cell_size)
    else:
        raise ValueError(f"Invalid format '{format}'. Use 'prettytable', 'html', or 'image'")


# ============================================================================
# DEMONSTRATION
# ============================================================================

def demo_simplified_output():
    """Demonstrate prettytable, HTML, and image outputs"""
    
    print("=" * 80)
    print("COMPARISON TABLE GENERATION - PRETTYTABLE FORMAT")
    print("=" * 80)
    
    # Example 1: PrettyTable to console (default)
    print("\n1. PrettyTable Output to Console")
    print("-" * 80)
    matrix1 = build_full_comparison_matrix(10, start_avg=3.0, end_avg=4.0, step_size=0.2)
    create_comparison_table(matrix1, output_file=None, format='prettytable')
    
    # Example 2: PrettyTable to file
    print("\n2. PrettyTable Output to File")
    print("-" * 80)
    matrix2 = build_full_comparison_matrix(15, start_avg=3.5, end_avg=4.5, step_size=0.1)
    txt_file = create_comparison_table(
        matrix2,
        os.path.join("..", "text", "comparison_table.txt"),
        format='prettytable'
    )
    
    # Example 3: PrettyTable with colors (console only)
    print("\n3. PrettyTable with Colors (if colorama available)")
    print("-" * 80)
    matrix3 = build_full_comparison_matrix(8, start_avg=3.5, end_avg=4.0, step_size=0.1)
    create_comparison_table(matrix3, output_file=None, format='prettytable', use_colors=True)
    
    # Example 4: Smaller example for better readability
    print("\n4. Small Example (better for console viewing)")
    print("-" * 80)
    matrix4 = build_full_comparison_matrix(10, start_avg=3.5, end_avg=4.0, step_size=0.25)
    create_comparison_table(matrix4, output_file=None, format='prettytable')
    
    # Example 5: Save another example to file
    print("\n5. Another Example Saved to File")
    print("-" * 80)
    matrix5 = build_full_comparison_matrix(20, start_avg=3.0, end_avg=5.0, step_size=0.2)
    txt_file2 = create_comparison_table(
        matrix5,
        os.path.join("..", "text", "comparison_large.txt"),
        format='prettytable'
    )
    
    print("\n" + "=" * 80)
    print("COMPLETE!")
    print("=" * 80)
    print("\nFiles created:")
    print(f"  • Text table: {txt_file}")
    print(f"  • Large text table: {txt_file2}")
    print("\nYou can:")
    print("  • View text files in any text editor")
    print("  • Copy/paste into emails, documents, or code")
    print("  • Print to console with format='prettytable' and output_file=None")
    print("  • Use use_colors=True for colored console output (if colorama installed)")
    print("\nOther formats still available:")
    print("  • format='html' for HTML output")
    print("  • format='image' for PNG/JPG output")


# if __name__ == "__main__":
#     demo_simplified_output()

In [23]:
demo_simplified_output()

COMPARISON TABLE GENERATION - PRETTYTABLE FORMAT

1. PrettyTable Output to Console
--------------------------------------------------------------------------------

Score Distribution Comparison Matrix
Population: 10 | Range: 3.0 to 4.0 | Step: 0.2

+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
| From↓/To→ |       3.0       |       3.2       |       3.4       |       3.6       |       3.8       |       4.0       |
+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|    3.0    |        —        |  4:+0.20(+2.0%) |  4:+0.38(+3.8%) |  4:+0.53(+5.3%) |  4:+0.63(+6.3%) |  4:+0.69(+6.9%) |
|           |                 |  5:+0.41(+4.1%) |  5:+0.87(+8.7%) | 5:+1.37(+13.7%) | 5:+1.93(+19.3%) | 5:+2.55(+25.5%) |
+-----------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|    3.2    |  4:-

In [25]:
matrix_data = build_full_comparison_matrix(n_respondents=15, 
                                           start_avg=3.0, 
                                           end_avg=5.0, 
                                           step_size=0.1)

In [27]:
create_comparison_table(matrix_data, 
                        output_file=os.path.join('..', 'images', 'comps_15_3_5_point1.png'), 
                        format='image', 
                        dpi=150, 
                        fig_width=None, 
                        fig_height=None, 
                        cell_size=None,
                        use_colors=False)

✓ Image saved to ../images/comps_15_3_5_point1.png


'../images/comps_15_3_5_point1.png'

In [29]:
create_comparison_table(matrix_data, 
                        output_file=os.path.join('..', 'html', 'comps_15_3_5_point1.html'), 
                        format='html', 
                        dpi=150, 
                        fig_width=None, 
                        fig_height=None, 
                        cell_size=None,
                        use_colors=False)

✓ HTML table saved to ../html/comps_15_3_5_point1.html


'../html/comps_15_3_5_point1.html'

In [50]:
create_comparison_table(matrix_data, 
                        output_file=os.path.join('..', 'text', 'comps_15_3_5_point1.txt'), 
                        format='prettytable', 
                        dpi=150, 
                        fig_width=None, 
                        fig_height=None, 
                        cell_size=None,
                        use_colors=False)

✓ Text table saved to ../text/comps_15_3_5_point1.txt


'../text/comps_15_3_5_point1.txt'

## Discontinuity or Jump Analysis
 - Where does a small difference in average score acutally make a large difference in the distribution?

In [31]:
from math import factorial
import numpy as np

# # Import the core functions
# import sys
# sys.path.insert(0, '/home/claude')
# from simplified_output import (
#     build_full_comparison_matrix,
#     calculate_average_distribution
# )

def analyze_distribution_jumps(matrix_data, threshold=0.5):
    """
    Identify "jumps" or discontinuities in score distributions.
    
    A "jump" occurs when a small change in average causes a large, 
    concentrated change in the distribution (e.g., one person moving from 3→2).
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    threshold : float
        Minimum absolute change to consider significant (default: 0.5)
    
    Returns:
    --------
    dict : Analysis results including jump points and metrics
    """
    averages = matrix_data["averages"]
    matrix = matrix_data["matrix"]
    n = matrix_data["population_size"]
    
    jump_points = []
    
    # Analyze consecutive pairs
    for i in range(len(averages) - 1):
        avg1 = averages[i]
        avg2 = averages[i + 1]
        
        cell_data = matrix[avg1][avg2]
        
        if cell_data["type"] == "valid":
            differences = cell_data["differences"]
            
            # Extract deltas for all scores
            deltas = [diff[0] for diff in differences]
            
            # Calculate metrics
            max_abs_change = max(abs(d) for d in deltas)
            max_change_score = deltas.index(max(deltas, key=abs)) + 1
            
            # Calculate concentration: what % of total change is in the max change?
            total_abs_change = sum(abs(d) for d in deltas)
            if total_abs_change > 0:
                concentration = abs(deltas[max_change_score - 1]) / total_abs_change
            else:
                concentration = 0
            
            # Calculate volatility: standard deviation of changes
            volatility = np.std(deltas)
            
            # Check if this is a "jump"
            # A jump is a large change that's concentrated in one or two scores
            is_jump = (max_abs_change >= threshold and concentration > 0.45)
            
            # Identify the pattern
            if is_jump:
                if max_change_score <= 2:
                    pattern = "Low score shift"
                elif max_change_score >= 4:
                    pattern = "High score shift"
                else:
                    pattern = "Mid score shift"
            else:
                pattern = "Gradual change"
            
            jump_info = {
                "from_avg": avg1,
                "to_avg": avg2,
                "step": avg2 - avg1,
                "max_change": max_abs_change,
                "max_change_score": max_change_score,
                "concentration": concentration,
                "volatility": volatility,
                "is_jump": is_jump,
                "pattern": pattern,
                "all_deltas": deltas
            }
            
            if is_jump:
                jump_points.append(jump_info)
    
    return {
        "population_size": n,
        "threshold": threshold,
        "jump_points": jump_points,
        "total_comparisons": len(averages) - 1
    }


def find_largest_single_score_changes(matrix_data, top_n=10):
    """
    Find the largest single-score changes in the entire range.
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    top_n : int
        Number of largest changes to return (default: 10)
    
    Returns:
    --------
    list : Sorted list of largest changes
    """
    averages = matrix_data["averages"]
    matrix = matrix_data["matrix"]
    
    all_changes = []
    
    for i in range(len(averages) - 1):
        avg1 = averages[i]
        avg2 = averages[i + 1]
        
        cell_data = matrix[avg1][avg2]
        
        if cell_data["type"] == "valid":
            differences = cell_data["differences"]
            
            for score_idx, (delta, pct) in enumerate(differences):
                all_changes.append({
                    "from_avg": avg1,
                    "to_avg": avg2,
                    "score": score_idx + 1,
                    "delta": delta,
                    "percent": pct,
                    "abs_delta": abs(delta)
                })
    
    # Sort by absolute change
    all_changes.sort(key=lambda x: x["abs_delta"], reverse=True)
    
    return all_changes[:top_n]


def calculate_change_smoothness(matrix_data):
    """
    Calculate how smooth or jumpy the transitions are across the entire range.
    
    Lower scores = smoother transitions
    Higher scores = more jumpy/discontinuous
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    
    Returns:
    --------
    dict : Smoothness metrics
    """
    averages = matrix_data["averages"]
    matrix = matrix_data["matrix"]
    
    all_volatilities = []
    all_concentrations = []
    
    for i in range(len(averages) - 1):
        avg1 = averages[i]
        avg2 = averages[i + 1]
        
        cell_data = matrix[avg1][avg2]
        
        if cell_data["type"] == "valid":
            differences = cell_data["differences"]
            deltas = [diff[0] for diff in differences]
            
            # Volatility (std deviation of changes)
            volatility = np.std(deltas)
            all_volatilities.append(volatility)
            
            # Concentration (how concentrated is change in one score?)
            total_abs_change = sum(abs(d) for d in deltas)
            if total_abs_change > 0:
                max_abs = max(abs(d) for d in deltas)
                concentration = max_abs / total_abs_change
                all_concentrations.append(concentration)
    
    # Handle edge case where no valid comparisons exist
    if not all_volatilities:
        return {
            "avg_volatility": 0,
            "max_volatility": 0,
            "avg_concentration": 0,
            "max_concentration": 0,
            "smoothness_score": 1.0,
            "note": "No valid comparisons found (all N/A)"
        }
    
    return {
        "avg_volatility": np.mean(all_volatilities),
        "max_volatility": np.max(all_volatilities),
        "avg_concentration": np.mean(all_concentrations) if all_concentrations else 0,
        "max_concentration": np.max(all_concentrations) if all_concentrations else 0,
        "smoothness_score": 1 - np.mean(all_concentrations) if all_concentrations else 1.0
    }


def print_jump_analysis(jump_analysis):
    """
    Pretty print the jump analysis results.
    
    Parameters:
    -----------
    jump_analysis : dict
        Output from analyze_distribution_jumps
    """
    print("\n" + "=" * 80)
    print("DISTRIBUTION JUMP ANALYSIS")
    print("=" * 80)
    
    print(f"\nPopulation Size: {jump_analysis['population_size']}")
    print(f"Jump Threshold: {jump_analysis['threshold']}")
    print(f"Total Comparisons: {jump_analysis['total_comparisons']}")
    print(f"Jump Points Found: {len(jump_analysis['jump_points'])}")
    
    if jump_analysis['jump_points']:
        print("\n" + "-" * 80)
        print("DETECTED JUMPS:")
        print("-" * 80)
        
        for i, jump in enumerate(jump_analysis['jump_points'], 1):
            print(f"\nJump #{i}:")
            print(f"  Average: {jump['from_avg']:.1f} → {jump['to_avg']:.1f} (step: {jump['step']:.2f})")
            print(f"  Pattern: {jump['pattern']}")
            print(f"  Max Change: {jump['max_change']:+.3f} in score {jump['max_change_score']}")
            print(f"  Concentration: {jump['concentration']:.1%} of change in one score")
            print(f"  Volatility: {jump['volatility']:.3f}")
            print(f"  All changes: ", end="")
            for score, delta in enumerate(jump['all_deltas'], 1):
                if abs(delta) > 0.01:
                    print(f"Score{score}:{delta:+.2f} ", end="")
            print()
    else:
        print("\nNo significant jumps detected.")
        print("The distribution changes are relatively smooth across this range.")
    
    print("\n" + "=" * 80)


def print_largest_changes(largest_changes):
    """
    Pretty print the largest single-score changes.
    
    Parameters:
    -----------
    largest_changes : list
        Output from find_largest_single_score_changes
    """
    print("\n" + "=" * 80)
    print("LARGEST SINGLE-SCORE CHANGES")
    print("=" * 80)
    
    print(f"\n{'Rank':<6} {'From→To':<15} {'Score':<7} {'Change':<15} {'% of Pop':<12} {'Type'}")
    print("-" * 80)
    
    for i, change in enumerate(largest_changes, 1):
        direction = "Increase" if change['delta'] > 0 else "Decrease"
        print(f"{i:<6} {change['from_avg']:.1f}→{change['to_avg']:.1f}{'':<7} "
              f"{change['score']:<7} {change['delta']:+8.3f}{'':<7} "
              f"{change['percent']:+6.1f}%{'':<6} {direction}")
    
    print("=" * 80)


def print_smoothness_analysis(smoothness):
    """
    Pretty print the smoothness analysis.
    
    Parameters:
    -----------
    smoothness : dict
        Output from calculate_change_smoothness
    """
    print("\n" + "=" * 80)
    print("DISTRIBUTION SMOOTHNESS ANALYSIS")
    print("=" * 80)
    
    print(f"\nAverage Volatility: {smoothness['avg_volatility']:.3f}")
    print(f"Maximum Volatility: {smoothness['max_volatility']:.3f}")
    print(f"Average Concentration: {smoothness['avg_concentration']:.1%}")
    print(f"Maximum Concentration: {smoothness['max_concentration']:.1%}")
    print(f"\nSmoothness Score: {smoothness['smoothness_score']:.3f}")
    
    # Interpretation
    score = smoothness['smoothness_score']
    if score > 0.7:
        interpretation = "Very smooth - changes are well-distributed across scores"
    elif score > 0.5:
        interpretation = "Moderately smooth - some concentrated changes"
    elif score > 0.3:
        interpretation = "Somewhat jumpy - notable concentrated changes"
    else:
        interpretation = "Very jumpy - changes highly concentrated in specific scores"
    
    print(f"Interpretation: {interpretation}")
    print("=" * 80)


# ============================================================================
# DEMONSTRATION
# ============================================================================

def demo_jump_detection():
    """Demonstrate jump detection on various scenarios"""
    
    print("=" * 80)
    print("DISTRIBUTION JUMP DETECTION DEMO")
    print("=" * 80)
    
    # Scenario 1: Fine granularity (more likely to have jumps)
    print("\n\nSCENARIO 1: Fine step size (0.1) with n=15")
    print("-" * 80)
    matrix1 = build_full_comparison_matrix(15, 3.5, 4.5, 0.1)
    
    jumps1 = analyze_distribution_jumps(matrix1, threshold=0.3)
    print_jump_analysis(jumps1)
    
    smoothness1 = calculate_change_smoothness(matrix1)
    print_smoothness_analysis(smoothness1)
    
    largest1 = find_largest_single_score_changes(matrix1, top_n=5)
    print_largest_changes(largest1)
    
    # Scenario 2: Coarse granularity (fewer, larger jumps)
    print("\n\nSCENARIO 2: Coarse step size (0.2) with n=20")
    print("-" * 80)
    matrix2 = build_full_comparison_matrix(20, 3.0, 5.0, 0.2)
    
    jumps2 = analyze_distribution_jumps(matrix2, threshold=0.5)
    print_jump_analysis(jumps2)
    
    smoothness2 = calculate_change_smoothness(matrix2)
    print_smoothness_analysis(smoothness2)
    
    largest2 = find_largest_single_score_changes(matrix2, top_n=5)
    print_largest_changes(largest2)
    
    # Scenario 3: Very fine granularity
    print("\n\nSCENARIO 3: Very fine step size (0.05) with n=20")
    print("-" * 80)
    matrix3 = build_full_comparison_matrix(20, 4.0, 4.5, 0.05)
    
    jumps3 = analyze_distribution_jumps(matrix3, threshold=0.2)
    print_jump_analysis(jumps3)
    
    smoothness3 = calculate_change_smoothness(matrix3)
    print_smoothness_analysis(smoothness3)


# if __name__ == "__main__":
#     demo_jump_detection()

In [33]:
demo_jump_detection()

DISTRIBUTION JUMP DETECTION DEMO


SCENARIO 1: Fine step size (0.1) with n=15
--------------------------------------------------------------------------------

DISTRIBUTION JUMP ANALYSIS

Population Size: 15
Jump Threshold: 0.3
Total Comparisons: 10
Jump Points Found: 0

No significant jumps detected.
The distribution changes are relatively smooth across this range.


DISTRIBUTION SMOOTHNESS ANALYSIS

Average Volatility: 0.000
Maximum Volatility: 0.000
Average Concentration: 0.0%
Maximum Concentration: 0.0%

Smoothness Score: 1.000
Interpretation: Very smooth - changes are well-distributed across scores

LARGEST SINGLE-SCORE CHANGES

Rank   From→To         Score   Change          % of Pop     Type
--------------------------------------------------------------------------------


SCENARIO 2: Coarse step size (0.2) with n=20
--------------------------------------------------------------------------------

DISTRIBUTION JUMP ANALYSIS

Population Size: 20
Jump Threshold: 0.5
Total Compariso

## Sensitivity Analysis

- Where does a small difference in the distribution make a large difference on the average score?

In [35]:
from math import factorial
import numpy as np

# # Import the core functions
# import sys
# sys.path.insert(0, '/home/claude')
# from simplified_output import (
#     build_full_comparison_matrix,
#     calculate_average_distribution,
#     find_distributions_with_average
# )

def calculate_marginal_impact(n_respondents, current_avg, from_score, to_score):
    """
    Calculate the impact on average when ONE person changes their score.
    
    Parameters:
    -----------
    n_respondents : int
        Total number of respondents
    current_avg : float
        Current average score
    from_score : int
        Score the person is changing FROM (1-5)
    to_score : int
        Score the person is changing TO (1-5)
    
    Returns:
    --------
    dict : Impact metrics
    """
    # Calculate the change in sum
    sum_change = to_score - from_score
    
    # Calculate the change in average
    avg_change = sum_change / n_respondents
    
    # Calculate the new average
    new_avg = current_avg + avg_change
    
    # Calculate percentage change in average
    if current_avg != 0:
        pct_change = (avg_change / current_avg) * 100
    else:
        pct_change = 0
    
    return {
        "n": n_respondents,
        "current_avg": current_avg,
        "from_score": from_score,
        "to_score": to_score,
        "avg_change": avg_change,
        "new_avg": new_avg,
        "pct_change": pct_change,
        "impact_magnitude": abs(avg_change)
    }


def analyze_all_single_person_impacts(n_respondents, current_avg):
    """
    Calculate impact of all possible single-person score changes.
    
    Parameters:
    -----------
    n_respondents : int
        Total number of respondents
    current_avg : float
        Current average score
    
    Returns:
    --------
    list : All possible impacts, sorted by magnitude
    """
    impacts = []
    
    for from_score in range(1, 6):
        for to_score in range(1, 6):
            if from_score != to_score:
                impact = calculate_marginal_impact(n_respondents, current_avg, 
                                                   from_score, to_score)
                impacts.append(impact)
    
    # Sort by impact magnitude (largest first)
    impacts.sort(key=lambda x: x["impact_magnitude"], reverse=True)
    
    return impacts


def calculate_distribution_stability(matrix_data):
    """
    Calculate stability (resistance to change) for each average in the range.
    
    Lower stability = more sensitive (small changes have big impact)
    Higher stability = less sensitive (small changes have small impact)
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    
    Returns:
    --------
    dict : Stability metrics for each average
    """
    averages = matrix_data["averages"]
    n = matrix_data["population_size"]
    
    stability_data = {}
    
    for avg in averages:
        # Calculate average impact of a single score change
        # We'll use the average impact of moving from adjacent scores
        impacts = []
        
        for from_score in range(1, 5):  # 1→2, 2→3, 3→4, 4→5
            to_score = from_score + 1
            impact = calculate_marginal_impact(n, avg, from_score, to_score)
            impacts.append(impact["avg_change"])
        
        avg_impact = np.mean([abs(i) for i in impacts])
        max_impact = max([abs(i) for i in impacts])
        
        # Stability is inverse of sensitivity
        # Higher impact = lower stability
        stability_score = 1 / (max_impact * 10) if max_impact > 0 else 100
        
        stability_data[avg] = {
            "average": avg,
            "avg_single_change_impact": avg_impact,
            "max_single_change_impact": max_impact,
            "stability_score": stability_score,
            "n": n
        }
    
    return stability_data


def find_most_sensitive_distributions(matrix_data, top_n=10):
    """
    Find the distributions (averages) most sensitive to single-person changes.
    
    Parameters:
    -----------
    matrix_data : dict
        Output from build_full_comparison_matrix
    top_n : int
        Number of most sensitive distributions to return
    
    Returns:
    --------
    list : Most sensitive distributions sorted by sensitivity
    """
    stability_data = calculate_distribution_stability(matrix_data)
    
    # Convert to list and sort by max impact (descending)
    sensitivity_list = []
    for avg, data in stability_data.items():
        sensitivity_list.append({
            "average": avg,
            "max_impact": data["max_single_change_impact"],
            "avg_impact": data["avg_single_change_impact"],
            "stability_score": data["stability_score"]
        })
    
    sensitivity_list.sort(key=lambda x: x["max_impact"], reverse=True)
    
    return sensitivity_list[:top_n]


def find_highest_leverage_changes(n_respondents, avg_range, top_n=10):
    """
    Find which specific score changes have the highest leverage (impact on average).
    
    Parameters:
    -----------
    n_respondents : int
        Number of respondents
    avg_range : tuple
        (start_avg, end_avg) to analyze
    top_n : int
        Number of highest leverage changes to return
    
    Returns:
    --------
    list : Highest leverage changes
    """
    start_avg, end_avg = avg_range
    
    # Sample several averages in the range
    num_samples = min(20, int((end_avg - start_avg) / 0.1) + 1)
    avg_samples = np.linspace(start_avg, end_avg, num_samples)
    
    all_leverage = []
    
    for avg in avg_samples:
        impacts = analyze_all_single_person_impacts(n_respondents, avg)
        
        for impact in impacts[:5]:  # Top 5 for each average
            all_leverage.append({
                "at_average": avg,
                "from_score": impact["from_score"],
                "to_score": impact["to_score"],
                "avg_change": impact["avg_change"],
                "impact_magnitude": impact["impact_magnitude"],
                "n": n_respondents
            })
    
    # Sort by impact magnitude
    all_leverage.sort(key=lambda x: x["impact_magnitude"], reverse=True)
    
    # Remove duplicates (same from→to change at different averages)
    seen = set()
    unique_leverage = []
    for item in all_leverage:
        key = (item["from_score"], item["to_score"])
        if key not in seen:
            seen.add(key)
            unique_leverage.append(item)
    
    return unique_leverage[:top_n]


def compare_population_sensitivity(averages_to_compare, populations):
    """
    Compare how population size affects sensitivity at specific averages.
    
    Parameters:
    -----------
    averages_to_compare : list
        List of averages to analyze (e.g., [3.0, 3.5, 4.0])
    populations : list
        List of population sizes to compare (e.g., [5, 10, 20, 50])
    
    Returns:
    --------
    dict : Comparison data
    """
    comparison = {}
    
    for avg in averages_to_compare:
        comparison[avg] = {}
        
        for n in populations:
            # Calculate impact of a 1-point change (e.g., 3→4)
            impact = calculate_marginal_impact(n, avg, 3, 4)
            comparison[avg][n] = {
                "avg_change": impact["avg_change"],
                "impact_magnitude": impact["impact_magnitude"]
            }
    
    return comparison


# ============================================================================
# PRINTING FUNCTIONS
# ============================================================================

def print_marginal_impacts(impacts, title="Single-Person Impact Analysis"):
    """Pretty print marginal impact analysis"""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    
    print(f"\n{'Rank':<6} {'Change':<12} {'Avg Impact':<15} {'New Avg':<12} {'% Change'}")
    print("-" * 80)
    
    for i, impact in enumerate(impacts[:10], 1):
        change_str = f"{impact['from_score']}→{impact['to_score']}"
        print(f"{i:<6} {change_str:<12} {impact['avg_change']:+.4f}{'':<10} "
              f"{impact['new_avg']:.4f}{'':<6} {impact['pct_change']:+.2f}%")
    
    print("=" * 80)


def print_sensitivity_analysis(sensitivity_list, title="Most Sensitive Distributions"):
    """Pretty print sensitivity analysis"""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    
    print(f"\n{'Rank':<6} {'Average':<10} {'Max Impact':<15} {'Avg Impact':<15} {'Sensitivity'}")
    print("-" * 80)
    
    for i, item in enumerate(sensitivity_list, 1):
        sensitivity = "HIGH" if item["max_impact"] > 0.15 else "MEDIUM" if item["max_impact"] > 0.10 else "LOW"
        print(f"{i:<6} {item['average']:<10.1f} {item['max_impact']:.4f}{'':<10} "
              f"{item['avg_impact']:.4f}{'':<10} {sensitivity}")
    
    print("\n" + "=" * 80)
    print("Interpretation:")
    print("• HIGH sensitivity: One person's score change moves average by >0.15")
    print("• MEDIUM sensitivity: One person's score change moves average by 0.10-0.15")
    print("• LOW sensitivity: One person's score change moves average by <0.10")
    print("=" * 80)


def print_leverage_analysis(leverage_list, title="Highest Leverage Score Changes"):
    """Pretty print leverage analysis"""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    
    print(f"\n{'Rank':<6} {'Change':<12} {'Impact':<15} {'At Average':<15} {'Population'}")
    print("-" * 80)
    
    for i, item in enumerate(leverage_list, 1):
        change_str = f"{item['from_score']}→{item['to_score']}"
        print(f"{i:<6} {change_str:<12} {item['avg_change']:+.4f}{'':<10} "
              f"{item['at_average']:.2f}{'':<12} n={item['n']}")
    
    print("\n" + "=" * 80)
    print("Key Insight: Larger score changes (1→5, 5→1) have the most leverage!")
    print("=" * 80)


def print_population_comparison(comparison):
    """Pretty print population size comparison"""
    print("\n" + "=" * 80)
    print("POPULATION SIZE SENSITIVITY COMPARISON")
    print("=" * 80)
    print("\nImpact of ONE person changing from score 3→4:\n")
    
    # Get populations and averages
    averages = sorted(comparison.keys())
    populations = sorted(list(comparison[averages[0]].keys()))
    
    # Print header
    print(f"{'Average':<12}", end="")
    for n in populations:
        print(f"n={n:<8}", end="")
    print()
    print("-" * 80)
    
    # Print data
    for avg in averages:
        print(f"{avg:<12.1f}", end="")
        for n in populations:
            impact = comparison[avg][n]["avg_change"]
            print(f"{impact:+.4f}{'':<4}", end="")
        print()
    
    print("\n" + "=" * 80)
    print("Key Insight: Smaller populations are MUCH more sensitive!")
    print("=" * 80)


# ============================================================================
# DEMONSTRATION
# ============================================================================

def demo_sensitivity_analysis():
    """Demonstrate sensitivity and leverage analysis"""
    
    print("=" * 80)
    print("DISTRIBUTION SENSITIVITY & LEVERAGE ANALYSIS DEMO")
    print("=" * 80)
    
    # Demo 1: Single population sensitivity
    print("\n\nDEMO 1: Impact Analysis for n=10")
    print("-" * 80)
    
    impacts_10 = analyze_all_single_person_impacts(10, 4.0)
    print_marginal_impacts(impacts_10, "What happens if ONE person changes their score? (n=10, avg=4.0)")
    
    # Demo 2: Compare different population sizes
    print("\n\nDEMO 2: Population Size Comparison")
    print("-" * 80)
    
    comparison = compare_population_sensitivity(
        averages_to_compare=[3.0, 3.5, 4.0, 4.5],
        populations=[5, 10, 20, 50, 100]
    )
    print_population_comparison(comparison)
    
    # Demo 3: Find most sensitive distributions
    print("\n\nDEMO 3: Most Sensitive Distributions")
    print("-" * 80)
    
    matrix = build_full_comparison_matrix(10, 3.0, 5.0, 0.2)
    sensitive = find_most_sensitive_distributions(matrix, top_n=5)
    print_sensitivity_analysis(sensitive)
    
    # Demo 4: Highest leverage changes
    print("\n\nDEMO 4: Highest Leverage Changes")
    print("-" * 80)
    
    leverage = find_highest_leverage_changes(10, (3.0, 5.0), top_n=10)
    print_leverage_analysis(leverage)
    
    # Demo 5: Small vs Large population
    print("\n\nDEMO 5: Small Population (n=5) - Very Sensitive!")
    print("-" * 80)
    
    matrix_small = build_full_comparison_matrix(5, 3.0, 4.5, 0.3)
    sensitive_small = find_most_sensitive_distributions(matrix_small, top_n=5)
    print_sensitivity_analysis(sensitive_small, "Sensitivity with n=5")
    
    impacts_5 = analyze_all_single_person_impacts(5, 4.0)
    print_marginal_impacts(impacts_5, "Single-Person Impact (n=5, avg=4.0)")
    
    print("\n\nDEMO 6: Large Population (n=50) - More Stable")
    print("-" * 80)
    
    matrix_large = build_full_comparison_matrix(50, 3.0, 4.5, 0.3)
    sensitive_large = find_most_sensitive_distributions(matrix_large, top_n=5)
    print_sensitivity_analysis(sensitive_large, "Sensitivity with n=50")
    
    impacts_50 = analyze_all_single_person_impacts(50, 4.0)
    print_marginal_impacts(impacts_50, "Single-Person Impact (n=50, avg=4.0)")


# if __name__ == "__main__":
#     demo_sensitivity_analysis()

In [56]:
demo_sensitivity_analysis()

DISTRIBUTION SENSITIVITY & LEVERAGE ANALYSIS DEMO


DEMO 1: Impact Analysis for n=10
--------------------------------------------------------------------------------

What happens if ONE person changes their score? (n=10, avg=4.0)

Rank   Change       Avg Impact      New Avg      % Change
--------------------------------------------------------------------------------
1      1→5          +0.4000           4.4000       +10.00%
2      5→1          -0.4000           3.6000       -10.00%
3      1→4          +0.3000           4.3000       +7.50%
4      2→5          +0.3000           4.3000       +7.50%
5      4→1          -0.3000           3.7000       -7.50%
6      5→2          -0.3000           3.7000       -7.50%
7      1→3          +0.2000           4.2000       +5.00%
8      2→4          +0.2000           4.2000       +5.00%
9      3→1          -0.2000           3.8000       -5.00%
10     3→5          +0.2000           4.2000       +5.00%


DEMO 2: Population Size Comparison
----------

## How Distributions Change When the Average Changes

In [37]:
from math import factorial
from itertools import product

def count_permutations_for_distribution(distribution, n):
    """Calculate number of permutations for a given distribution using multinomial coefficient"""
    numerator = factorial(n)
    denominator = 1
    for count in distribution:
        denominator *= factorial(count)
    return numerator // denominator


def find_distributions_with_average(n_respondents, target_average, scores=[1,2,3,4,5]):
    """Find all score distributions that achieve target average"""
    
    # Check if average is achievable
    target_sum = target_average * n_respondents
    if target_sum != int(target_sum):
        return None, f"Average {target_average} is not possible with {n_respondents} respondents (sum must be integer)"
    
    target_sum = int(target_sum)
    
    # Check if sum is within valid range
    min_possible_sum = n_respondents * min(scores)
    max_possible_sum = n_respondents * max(scores)
    
    if target_sum < min_possible_sum or target_sum > max_possible_sum:
        return None, f"Average {target_average} is not achievable (sum {target_sum} out of range [{min_possible_sum}, {max_possible_sum}])"
    
    valid_distributions = []
    
    # Generate all possible distributions
    def generate_distributions(remaining_people, remaining_sum, current_dist, score_index):
        # Base case: we've assigned all scores
        if score_index == len(scores):
            if remaining_people == 0 and remaining_sum == 0:
                valid_distributions.append(tuple(current_dist))
            return
        
        current_score = scores[score_index]
        
        # Try all possible counts for current score
        max_count = min(remaining_people, remaining_sum // current_score if current_score > 0 else remaining_people)
        
        for count in range(max_count + 1):
            generate_distributions(
                remaining_people - count,
                remaining_sum - (count * current_score),
                current_dist + [count],
                score_index + 1
            )
    
    generate_distributions(n_respondents, target_sum, [], 0)
    
    return valid_distributions, None


def find_all_distribution_transitions(n_respondents, initial_avg, final_avg):
    """
    Find all possible changes in distribution counts when moving from 
    initial average to final average.
    
    Parameters:
    -----------
    n_respondents : int
        Number of respondents/people in the survey
    initial_avg : float
        Initial average score
    final_avg : float
        Final average score
    
    Returns:
    --------
    dict : Contains:
        - 'n': number of respondents
        - 'initial_avg': initial average
        - 'final_avg': final average
        - 'initial_distributions': list of all valid initial distributions
        - 'final_distributions': list of all valid final distributions
        - 'transitions': dict mapping each delta pattern to information about it
        - 'all_deltas': list of all unique delta patterns found
    """
    
    # Find all distributions for both averages
    initial_dists, error1 = find_distributions_with_average(n_respondents, initial_avg)
    final_dists, error2 = find_distributions_with_average(n_respondents, final_avg)
    
    if error1:
        return {'error': error1}
    if error2:
        return {'error': error2}
    
    # Calculate all possible transitions
    transitions = {}  # Maps delta tuple to info about that transition
    
    for init_dist in initial_dists:
        for final_dist in final_dists:
            # Calculate the change (delta) for each score
            delta = tuple(final_dist[i] - init_dist[i] for i in range(5))
            
            # Initialize this delta pattern if we haven't seen it
            if delta not in transitions:
                transitions[delta] = {
                    'delta': delta,
                    'total_paths': 0,
                    'examples': []
                }
            
            # Count how many paths lead to this delta
            # (weighted by permutations from both distributions)
            init_perms = count_permutations_for_distribution(init_dist, n_respondents)
            final_perms = count_permutations_for_distribution(final_dist, n_respondents)
            transitions[delta]['total_paths'] += init_perms * final_perms
            
            # Store an example if we don't have too many already
            if len(transitions[delta]['examples']) < 3:
                transitions[delta]['examples'].append({
                    'initial': init_dist,
                    'final': final_dist,
                    'initial_perms': init_perms,
                    'final_perms': final_perms
                })
    
    # Sort deltas by total paths (most common first)
    all_deltas = sorted(transitions.keys(), 
                       key=lambda d: transitions[d]['total_paths'], 
                       reverse=True)
    
    return {
        'n': n_respondents,
        'initial_avg': initial_avg,
        'final_avg': final_avg,
        'initial_distributions': initial_dists,
        'final_distributions': final_dists,
        'transitions': transitions,
        'all_deltas': all_deltas
    }


def print_transition_summary(result):
    """Print a summary of the transition analysis"""
    
    if 'error' in result:
        print(f"Error: {result['error']}")
        return
    
    n = result['n']
    initial_avg = result['initial_avg']
    final_avg = result['final_avg']
    
    print(f"\n{'='*80}")
    print(f"TRANSITION ANALYSIS: {initial_avg} → {final_avg} (n={n})")
    print(f"{'='*80}\n")
    
    print(f"Number of initial distributions: {len(result['initial_distributions'])}")
    print(f"Number of final distributions: {len(result['final_distributions'])}")
    print(f"Number of unique change patterns: {len(result['all_deltas'])}")
    
    print(f"\n{'='*80}")
    print("TOP CHANGE PATTERNS (by likelihood):")
    print(f"{'='*80}\n")
    
    # Show top 10 patterns
    for i, delta in enumerate(result['all_deltas'][:10], 1):
        info = result['transitions'][delta]
        print(f"\nPattern #{i}:")
        print(f"  Change: ", end="")
        for score, change in enumerate(delta, 1):
            if change != 0:
                print(f"Score {score}: {change:+d}  ", end="")
        print()
        print(f"  Total paths: {info['total_paths']:,}")
        print(f"  Example: {info['examples'][0]['initial']} → {info['examples'][0]['final']}")


def analyze_minimal_changes(result):
    """Find transitions that require the minimum number of people to change scores"""
    
    if 'error' in result:
        return
    
    print(f"\n{'='*80}")
    print("MINIMAL CHANGE ANALYSIS:")
    print(f"{'='*80}\n")
    
    # Group deltas by total number of people who changed
    by_change_count = {}
    
    for delta, info in result['transitions'].items():
        # Count how many people's scores changed (sum of positive changes)
        total_changes = sum(abs(d) for d in delta) // 2  # Divide by 2 since each change is counted twice
        
        if total_changes not in by_change_count:
            by_change_count[total_changes] = []
        
        by_change_count[total_changes].append((delta, info))
    
    # Show the patterns with minimal changes
    min_changes = min(by_change_count.keys())
    
    print(f"Minimum number of people who must change scores: {min_changes}\n")
    
    patterns = sorted(by_change_count[min_changes], 
                     key=lambda x: x[1]['total_paths'], 
                     reverse=True)
    
    for i, (delta, info) in enumerate(patterns[:5], 1):
        print(f"\nMinimal Pattern #{i}:")
        print(f"  Change: ", end="")
        for score, change in enumerate(delta, 1):
            if change != 0:
                print(f"Score {score}: {change:+d}  ", end="")
        print()
        print(f"  Paths: {info['total_paths']:,}")
        print(f"  Example: {info['examples'][0]['initial']} → {info['examples'][0]['final']}")


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # Example 1: Small change
    print("\n" + "="*80)
    print("EXAMPLE 1: Moving from 3.5 to 3.7 (n=10)")
    print("="*80)
    
    result1 = find_all_distribution_transitions(10, 3.5, 3.7)
    print_transition_summary(result1)
    analyze_minimal_changes(result1)
    
    # Example 2: Larger change
    print("\n\n" + "="*80)
    print("EXAMPLE 2: Moving from 3.0 to 4.0 (n=10)")
    print("="*80)
    
    result2 = find_all_distribution_transitions(10, 3.0, 4.0)
    print_transition_summary(result2)
    analyze_minimal_changes(result2)
    
    # Example 3: Small population
    print("\n\n" + "="*80)
    print("EXAMPLE 3: Moving from 3.0 to 4.0 (n=5)")
    print("="*80)
    
    result3 = find_all_distribution_transitions(5, 3.0, 4.0)
    print_transition_summary(result3)
    analyze_minimal_changes(result3)


EXAMPLE 1: Moving from 3.5 to 3.7 (n=10)

TRANSITION ANALYSIS: 3.5 → 3.7 (n=10)

Number of initial distributions: 43
Number of final distributions: 35
Number of unique change patterns: 310

TOP CHANGE PATTERNS (by likelihood):


Pattern #1:
  Change: Score 3: -1  Score 5: +1  
  Total paths: 6,556,209,120
  Example: (0, 0, 5, 5, 0) → (0, 0, 4, 5, 1)

Pattern #2:
  Change: Score 2: -1  Score 4: +1  
  Total paths: 6,155,803,620
  Example: (0, 1, 3, 6, 0) → (0, 0, 3, 7, 0)

Pattern #3:
  Change: Score 1: -1  Score 3: +1  
  Total paths: 5,701,382,100
  Example: (1, 0, 2, 7, 0) → (0, 0, 3, 7, 0)

Pattern #4:
  Change: Score 2: -1  Score 3: +1  Score 4: -1  Score 5: +1  
  Total paths: 5,093,992,800
  Example: (0, 1, 3, 6, 0) → (0, 0, 4, 5, 1)

Pattern #5:
  Change: Score 1: -1  Score 2: +1  Score 4: -1  Score 5: +1  
  Total paths: 4,621,762,800
  Example: (1, 0, 2, 7, 0) → (0, 1, 2, 6, 1)

Pattern #6:
  Change: Score 1: -1  Score 2: +1  Score 3: -1  Score 4: +1  
  Total paths: 4,325,41

## ROUGH WORK